# The Agentic Loop

### Basic request

In [1]:
import json
import os

from anthropic import Anthropic
from dotenv import load_dotenv

# ============================================================
# Configuration
# ============================================================
load_dotenv()

MODEL = "claude-haiku-4-5-20251001"

api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    raise RuntimeError("ANTHROPIC_API_KEY is not set")

client = Anthropic(api_key=api_key)

# ============================================================
# Send request to Claude
# ============================================================

response = client.messages.create(
    model=MODEL,
    max_tokens=20,
    messages=[
        {
            "role": "user",
            "content": "Say hello in one word",
        }
    ],
)

# Dump full response as JSON
print(json.dumps(response.model_dump(), indent=2))

{
  "id": "msg_011CefSTrzFY7ziuZRu6tVPZ",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "Hello",
      "type": "text"
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "not_available",
    "input_tokens": 12,
    "output_tokens": 4,
    "output_tokens_details": null,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}


```json
{
  "id": "msg_011CefSTrzFY7ziuZRu6tVPZ",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "Hello",
      "type": "text"
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "not_available",
    "input_tokens": 12,
    "output_tokens": 4,
    "output_tokens_details": null,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}
```

### Formatted output

In [2]:
from pydantic import BaseModel
from anthropic import Anthropic

class ContactInfo(BaseModel):
    name: str
    email: str
    plan_interest: str
    demo_requested: bool

client = Anthropic()

response = client.messages.parse(
    model=MODEL,
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Extract the key info: John Smith (john@example.com) wants the Enterprise plan and a demo next Tuesday.",
        }
    ],
    output_format=ContactInfo,
)

print(json.dumps(response.model_dump(), indent=2))

contact = response.parsed_output
print(contact.name, contact.email, contact.plan_interest, contact.demo_requested)

{
  "id": "msg_011CefT5pM1Z5PCpnqDas8Lz",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "{\"name\":\"John Smith\",\"email\":\"john@example.com\",\"plan_interest\":\"Enterprise\",\"demo_requested\":true}",
      "type": "text",
      "parsed_output": {
        "name": "John Smith",
        "email": "john@example.com",
        "plan_interest": "Enterprise",
        "demo_requested": true
      }
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "not_available",
    "input_tokens": 290,
    "output_tokens": 29,
    "output_tokens_details": null,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}
John Smith john

/home/dali/WORK/AInDrahim/ccarf/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed_output', input_value=ContactInfo(name='John Sm...e', demo_requested=True), input_type=ContactInfo])
  PydanticSerializationUnexpectedValue(Expected `ThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](...', demo_requested=True)), input_type=ParsedTextBlock[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `RedactedThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](...', demo_requested=True)), input_type=ParsedTextBlock[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `ToolUseBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](...', demo_r

### Agent loop

```python
while True:

    # 1. Ask the model what to do
    response = llm(messages)

    # 2. Check if the model wants to use a tool
    if response.tool_use:

        # 3. Execute the tool
        result = execute_tool(response.tool_use)

        # 4. Give the result back to the model
        messages.append(response)
        messages.append(result)

    else:
        # 5. Model has finished
        print(response.text)
        break
```

In [12]:
import json
import os

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-haiku-4-5-20251001"

api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    raise RuntimeError("ANTHROPIC_API_KEY is not set")

client = Anthropic(api_key=api_key)

def add_numbers(a, b):
    return a + b


tools = [
    {
        "name": "add_numbers",
        "description": "Add two numbers together",
        "input_schema": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "The first number"
                },
                "b": {
                    "type": "number",
                    "description": "The second number"
                },
            },
            "required": ["a", "b"],
        },
    }
]

messages = [
    {
        "role": "user",
        "content": "Find the answer to: What is seventeen plus twenty four",
    }
]

i=0
while True:
    i += 1
    print(f"**** loop :{i}")
    
    print (messages)
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        tools=tools,
        messages=messages,
    )

    # Add Claude's response to the conversation
    messages.append({
        "role": "assistant",
        "content": response.content,
    })

    # Claude has finished
    if response.stop_reason == "end_turn":
        print("-- Claude response")
        print(response.content[0].text)
        break

    # Claude wants to use a tool
    if response.stop_reason == "tool_use":
        for block in response.content:
            if block.type == "tool_use":
                print(f"Tool: {block.name}")
                print(f"Input: {block.input}")

                # Execute the requested tool
                if block.name == "add_numbers":
                    result = add_numbers(
                        block.input["a"],
                        block.input["b"]
                    )

                # Send tool result back to Claude
                messages.append({
                    "role": "user",
                    "content": [
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": str(result),
                        }
                    ],
                })


**** loop :1
[{'role': 'user', 'content': 'Find the answer to: What is seventeen plus twenty four'}]
Tool: add_numbers
Input: {'a': 17, 'b': 24}
**** loop :2
[{'role': 'user', 'content': 'Find the answer to: What is seventeen plus twenty four'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_013KbiswywQXYx371GZJM6dy', caller=DirectCaller(type='direct'), input={'a': 17, 'b': 24}, name='add_numbers', type='tool_use', toolset_name=None)]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_013KbiswywQXYx371GZJM6dy', 'content': '41'}]}]
-- Claude response
Seventeen plus twenty four equals **41**.


### Agent loop

- Claude's tool request comes from assistant.
- Execute the requested tool in your application.
- Return the result as a user message containing tool_result.
- Every tool_result must contain the matching tool_use_id, and don't mix ordinary text into that message.

In [14]:
import json
import os

from anthropic import Anthropic
from dotenv import load_dotenv


# ============================================================
# Configuration
# ============================================================

load_dotenv()

MODEL = "claude-haiku-4-5-20251001"

api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    raise RuntimeError("ANTHROPIC_API_KEY is not set")

client = Anthropic(api_key=api_key)


# ============================================================
# Data
# ============================================================

data = {
    "customers": [
        {
            "id": "CUS0001",
            "name": "Alice Johnson",
            "city": "Toronto"
        },
        {
            "id": "CUS0002",
            "name": "Bob Smith",
            "city": "Montreal"
        }
    ],
    "orders": [
        {
            "id": "ORD0001",
            "customer_id": "CUS0001",
            "product": "Laptop",
            "amount": 1200.00
        },
        {
            "id": "ORD0002",
            "customer_id": "CUS0001",
            "product": "Wireless Mouse",
            "amount": 35.50
        },
        {
            "id": "ORD0003",
            "customer_id": "CUS0002",
            "product": "Keyboard",
            "amount": 85.00
        },
        {
            "id": "ORD0004",
            "customer_id": "CUS0002",
            "product": "Monitor",
            "amount": 450.00
        }
    ]
}


# ============================================================
# Functions
# ============================================================

def get_customer_by_id(customer_id):
    return next(
        (
            customer
            for customer in data["customers"]
            if customer["id"] == customer_id
        ),
        None
    )


def get_order_by_customer_id(customer_id):
    return [
        order
        for order in data["orders"]
        if order["customer_id"] == customer_id
    ]


# ============================================================
# Claude Tools
# ============================================================

tools = [
    {
        "name": "get_customer_by_id",
        "description": "Get a customer by ID",
        "input_schema": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "string",
                    "description": "The customer ID, for example CUS0001"
                }
            },
            "required": ["id"],
        },
    },
    {
        "name": "get_order_by_customer_id",
        "description": "Get all orders for a customer by customer ID",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {
                    "type": "string",
                    "description": "The customer ID, for example CUS0001"
                }
            },
            "required": ["customer_id"],
        },
    }
]


# ============================================================
# Tool Executor
# ============================================================

def execute_tool(name, tool_input):

    if name == "get_customer_by_id":
        customer_id = tool_input.get("id")

        if not customer_id:
            return {
                "error": "Missing customer ID"
            }

        result = get_customer_by_id(customer_id)

        if result is None:
            return {
                "error": f"Customer {customer_id} not found"
            }

        return result

    if name == "get_order_by_customer_id":
        customer_id = tool_input.get("customer_id")

        if not customer_id:
            return {
                "error": "Missing customer ID"
            }

        return get_order_by_customer_id(customer_id)

    return {
        "error": f"Unknown tool: {name}"
    }


# ============================================================
# Agent
# ============================================================

def run_agent(user_message):

    messages = [
        {
            "role": "user",
            "content": user_message
        }
    ]

    while True:

        # ----------------------------------------------------
        # Call Claude
        # ----------------------------------------------------

        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        # ----------------------------------------------------
        # Add Claude's response to conversation
        # ----------------------------------------------------

        messages.append({
            "role": "assistant",
            "content": response.content
        })

        # ----------------------------------------------------
        # Claude finished
        # ----------------------------------------------------

        if response.stop_reason == "end_turn":

            for block in response.content:
                if block.type == "text":
                    return block.text

        # ----------------------------------------------------
        # Claude wants to use a tool
        # ----------------------------------------------------

        if response.stop_reason == "tool_use":

            tool_results = []

            for block in response.content:

                if block.type == "tool_use":

                    print(
                        f"\n[Tool call] {block.name}"
                    )

                    print(
                        f"[Input] {block.input}"
                    )

                    # Execute the Python function
                    result = execute_tool(
                        block.name,
                        block.input
                    )

                    print(
                        f"[Result] {result}"
                    )

                    # Create tool_result message
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)
                    })

            # ------------------------------------------------
            # Send tool results back to Claude
            # ------------------------------------------------

            messages.append({
                "role": "user",
                "content": tool_results
            })

            # Go back to Claude
            continue

        # ----------------------------------------------------
        # Unexpected response
        # ----------------------------------------------------

        raise RuntimeError(
            f"Unexpected stop reason: {response.stop_reason}"
        )

In [15]:
question = "give me customer CUS0001" 
answer = run_agent(question)
print(f"\nClaude: {answer}")


[Tool call] get_customer_by_id
[Input] {'id': 'CUS0001'}
[Result] {'id': 'CUS0001', 'name': 'Alice Johnson', 'city': 'Toronto'}

Claude: Here's the customer information for CUS0001:

- **ID**: CUS0001
- **Name**: Alice Johnson
- **City**: Toronto


In [16]:
question = "give me all orders of customer CUS0001" 
answer = run_agent(question)
print(f"\nClaude: {answer}")


[Tool call] get_order_by_customer_id
[Input] {'customer_id': 'CUS0001'}
[Result] [{'id': 'ORD0001', 'customer_id': 'CUS0001', 'product': 'Laptop', 'amount': 1200.0}, {'id': 'ORD0002', 'customer_id': 'CUS0001', 'product': 'Wireless Mouse', 'amount': 35.5}]

Claude: Here are all the orders for customer CUS0001:

1. **Order ID: ORD0001**
   - Product: Laptop
   - Amount: $1,200.00

2. **Order ID: ORD0002**
   - Product: Wireless Mouse
   - Amount: $35.50

**Total Orders: 2**
**Combined Amount: $1,235.50**


### Hub and spoke

- Coordinator = thinks/orchestrates; subagents = execute.
- Subagents have isolated contexts.
- Pass context explicitly through the Agent prompt.
- Never pass unnecessary full history.
- Independent agents → parallel Agent calls.
- Dependent agents → sequential calls.
- Use dynamic selection instead of always invoking every agent.
- Coordinator owns routing, aggregation, and error handling.

In [19]:
import os
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
client = Anthropic()

MODEL = "claude-haiku-4-5-20251001"


# --- Specialist agents: each is just a call with its own system prompt ---

def weather_agent(question):
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        system="You are a weather expert. Answer briefly and only about weather.",
        messages=[{"role": "user", "content": question}],
    )
    return response.content[0].text


def billing_agent(question):
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        system="You are a billing support agent. Answer briefly and only about billing.",
        messages=[{"role": "user", "content": question}],

    
    
    
    )
    return response.content[0].text


def general_agent(question):
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        system="You are a helpful general assistant.",
        messages=[{"role": "user", "content": question}],
    )
    return response.content[0].text


AGENTS = {
    "weather": weather_agent,
    "billing": billing_agent,
    "general": general_agent,
}


# --- Coordinator: decides which agent should handle the question ---

def coordinator(question):
    response = client.messages.create(
        model=MODEL,
        max_tokens=10,
        system=(
            "Classify the user's question into exactly one word: "
            "'weather', 'billing', or 'general'. Reply with only that word."
        ),
        messages=[{"role": "user", "content": question}],
    )
    choice = response.content[0].text.strip().lower()
    return choice if choice in AGENTS else "general"


def run(question):
    agent_name = coordinator(question)
    print(f"[coordinator routed to: {agent_name}]")
    answer = AGENTS[agent_name](question)
    return answer


# --- Try it ---
question = "Will it rain in Tunis tomorrow?"
#question = "Why was I charged twice this month?"
#question = "What's the capital of France?"

print("Agent:", run(question))

[coordinator routed to: weather]
Agent: I don't have access to current weather data or forecasts. To find out if it will rain in Tunis tomorrow, I recommend checking:

- **Weather.com** or **Weather.gov**
- **AccuWeather**
- **Local Tunisian meteorological services**
- Your phone's weather app

These sources provide real-time forecasts for Tunis.
